In [3]:
import json
import re
from pathlib import Path

import pandas as pd
import pymupdf

ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "data/documents.jsonl").exists())
DOCS = [json.loads(line) for line in open(ROOT / "data/documents.jsonl", encoding="utf-8")]
FILES = {d["doc_id"]: ROOT / "data/reports" / d["path"] for d in DOCS}


def letters_count(text):
    return len(re.findall(r"[А-Яа-яЁёA-Za-z]", text))


def inspect_pdf(path):
    """Сколько страниц и на скольких из них простая библиотека достаёт текст."""
    with pymupdf.open(path) as doc:
        with_text = sum(letters_count(page.get_text()) > 50 for page in doc)
        return len(doc), with_text


rows = []
for key, path in FILES.items():
    pages, with_text = inspect_pdf(path)
    rows.append({"doc_id": key, "размер, МБ": round(path.stat().st_size / 2**20, 1),
                 "страниц": pages, "с текстом": with_text, "без текста": pages - with_text})

pd.DataFrame(rows).sort_values("без текста", ascending=False)

,doc_id,"размер, МБ",страниц,с текстом,без текста
2,alfa_2024_annual,6.0,130,0,130
14,gpb_2025_annual,10.7,93,0,93
12,gpb_2023_annual,22.2,48,0,48
0,alfa_2022_annual,6.5,125,114,11
13,gpb_2024_annual,3.5,51,44,7
1,alfa_2023_annual,3.2,130,123,7
3,alfa_2025_annual,2.9,126,120,6
11,gpb_2022_annual,1.6,48,42,6
40,sovkom_2024_annual,4.2,135,131,4
41,sovkom_2025_6m,3.5,59,55,4
